## Open notebook in:
| Colab                                 
:-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nicolepcx/transformers-the-definitive-guide/blob/master/CH11/ch11_langgraph_code_interpreter.ipynb)                                             

# About This Notebook

This notebook shows you how to build a small agentic workflow using LangGraph, structured outputs, and the [E2B sandbox](https://e2b.dev/). You create a multi-step loop where one agent writes code, a non-LLM tester executes it inside a secure sandbox, and a second agent fixes the code based on structured feedback. The cycle repeats until the tests pass or the system reaches a set iteration limit.

You start with a clean state definition and structured artifacts. `CodeDraft` carries the generated Python implementation. `TestResult` carries the feedback. `FibState` keeps the full global conversation and execution state across nodes. Each node focuses on one responsibility and nothing else. The sandbox isolates execution from your environment so you can safely test arbitrary generated code.

Treat this notebook as a blueprint. Swap the Fibonacci harness for a domain specific test suite, adjust the iteration limits, and point the graph at a different objective. The pattern stays the same. Create, evaluate, improve. Build systems that learn from their own output.

<br>

__Note: You'll need an API from E2B.__


# Dependencies

In [1]:
%pip install langgraph e2b_code_interpreter langchain langchainhub langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.7/84.7 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.0/204.0 kB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 484.9/484.9 kB 41.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 243.4/243.4 kB 15.1 MB/s eta 0:00:00
  Attempting uninstall: packaging
    Found existing installation: packaging 25.0
    Uninstalling packaging-25.0:
      Successfully uninstalled packaging-25.0
  Attempting uninstall: rich
    Found existing installation: rich 13.9.4
    Uninstalling rich-13.9.4:
      Successfully uninstalled rich-13.9.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.1
    Uninstalling langchain-core-1.2.1:
      Successfully uninstalled langchain-core-1.2.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. Th

In [2]:
# Imports for API
from dotenv import load_dotenv
import os

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
E2B_API_KEY = os.getenv("E2B_API_KEY")


In [3]:
import os
import json
from typing import Optional, Dict, Any, List, TypedDict

from pydantic.v1 import BaseModel, Field
from e2b_code_interpreter import Sandbox

from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage, BaseMessage

from langgraph.graph import StateGraph, START, END

# Set LLMs

In [4]:
MODEL_NAME = "gpt-4.1-mini"

coder_llm = ChatOpenAI(model=MODEL_NAME, temperature=0)
fixer_llm = ChatOpenAI(model=MODEL_NAME, temperature=0)

# Structured payloads passed between agents

In [5]:
class CodeDraft(BaseModel):
    """Structured output from the code writer or fixer."""
    code: str = Field(description="Pure Python code that defines a Fibonacci function.")
    rationale: str = Field(description="Short explanation of the design choices.")


class TestResult(BaseModel):
    """Structured result from the tester."""
    passed: bool
    details: str
    failed_cases: List[str] = Field(default_factory=list)


class FibState(TypedDict, total=False):
    """Global LangGraph state. The key part is that it carries structured artifacts."""
    messages: List[BaseMessage]
    code: Optional[str]
    draft_rationale: Optional[str]
    tests: Optional[TestResult]
    iteration: int
    max_iterations: int
    done: bool

# Simple Fibonacci test harness using E2B sandbox

In [6]:
FIB_EXPECTED = [0, 1, 1, 2, 3, 5, 8, 13, 21, 34]


def create_sandbox() -> Sandbox:
    api_key = os.environ.get("E2B_API_KEY", "")
    if not api_key:
        raise RuntimeError("E2B_API_KEY is not set")
    # Modern E2B: use Sandbox.create(), not Sandbox()
    return Sandbox.create()


def run_fib_tests(sandbox: Sandbox, user_code: str) -> TestResult:
    """
    Takes the code from the agent, adds a tiny test harness,
    and runs it inside the E2B sandbox.
    """

    harness = f"""
import json

# User code comes first
{user_code}

results = {{}}

try:
    # Try to find a Fibonacci like function
    fib_fn = None
    for name in ("fib", "fibonacci", "Fib", "Fibonacci"):
        obj = globals().get(name)
        if callable(obj):
            fib_fn = obj
            break

    failed = []

    if fib_fn is None:
        failed.append("No fib function found")
    else:
        expected = {FIB_EXPECTED!r}
        got = [fib_fn(i) for i in range(len(expected))]
        if got != expected:
            failed.append(f"Wrong sequence: expected {{expected}}, got {{got}}")

    if failed:
        out = {{"passed": False, "failed_cases": failed}}
    else:
        out = {{"passed": True, "failed_cases": []}}

except Exception as e:
    out = {{"passed": False, "failed_cases": [f'Exception: {{repr(e)}}']}}

print(json.dumps(out))
"""

    execution = sandbox.run_code(harness)

    # Helper to normalize E2B outputs to plain text
    def _as_text(x: Any) -> str:
        if x is None:
            return ""
        if isinstance(x, str):
            return x
        if isinstance(x, list):
            return "\n".join(str(line) for line in x)
        return str(x)

    stdout = _as_text(execution.logs.stdout)
    stderr = _as_text(execution.logs.stderr)
    error = execution.error
    if isinstance(error, list):
        error = " ".join(str(e) for e in error)

    if error:
        return TestResult(
            passed=False,
            details=f"Sandbox error: {error}",
            failed_cases=[stderr.strip()] if stderr.strip() else [],
        )

    # Parse the last JSON line from stdout
    last_line = ""
    for line in stdout.splitlines():
        if line.strip():
            last_line = line.strip()

    try:
        data = json.loads(last_line)
        return TestResult(
            passed=bool(data.get("passed", False)),
            details="Tests completed",
            failed_cases=list(data.get("failed_cases", [])),
        )
    except Exception as e:
        return TestResult(
            passed=False,
            details=f"Could not parse test output: {e}",
            failed_cases=[stdout, stderr],
        )


# Agent nodes

In [7]:
WRITER_SYSTEM_PROMPT = (
    "You are a Python coding agent. Your task is to implement a Fibonacci function.\n"
    "Requirements:\n"
    "  • Define a function fib(n: int) -> int that returns the nth Fibonacci number\n"
    "  • Use an efficient iterative approach (no naive recursion)\n"
    "  • Handle n = 0 correctly\n"
    "Return your answer as a structured JSON object with fields 'code' and 'rationale'.\n"
    "The 'code' field must contain only valid Python and must define fib.\n"
)

FIXER_SYSTEM_PROMPT = (
    "You are a Python code fixer. You receive the previous code and test results.\n"
    "Requirements:\n"
    "  • First ensure correctness for all tests\n"
    "  • Keep the contract fib(n: int) -> int\n"
    "  • Prefer an iterative or fast doubling implementation\n"
    "Return a structured JSON object with fields 'code' and 'rationale'.\n"
)


def writer_node(state: FibState) -> FibState:
    """
    First agent: produce an initial Fibonacci implementation
    using structured output (CodeDraft).
    """
    messages = state["messages"]

    structured = coder_llm.with_structured_output(CodeDraft)  # Pydantic binding
    draft: CodeDraft = structured.invoke(
        [
            SystemMessage(content=WRITER_SYSTEM_PROMPT),
            *messages,
        ]
    )

    new_messages: List[BaseMessage] = list(messages) + [
        HumanMessage(content="Generated initial Fibonacci implementation."),
    ]

    return {
        "messages": new_messages,
        "code": draft.code,
        "draft_rationale": draft.rationale,
        "iteration": state.get("iteration", 0),
        "max_iterations": state.get("max_iterations", 3),
        "tests": None,
        "done": False,
    }


def tester_node(state: FibState, sandbox: Sandbox) -> FibState:
    """
    Non LLM testing harness. It receives `code` and returns a TestResult.
    This is a structured artifact, not just text.
    """
    code = state.get("code") or ""
    tests = run_fib_tests(sandbox, code)

    details = tests.details
    if tests.failed_cases:
        details += " | " + " | ".join(tests.failed_cases)

    new_messages: List[BaseMessage] = list(state["messages"]) + [
        HumanMessage(content=f"[tester] passed={tests.passed}, details={details}")
    ]

    return {
        **state,
        "messages": new_messages,
        "tests": tests,
    }


def fixer_node(state: FibState) -> FibState:
    """
    Second LLM agent that takes both the previous code and structured test feedback,
    and produces a refined CodeDraft.
    """
    code = state.get("code") or ""
    tests = state.get("tests")

    tests_json = tests.model_dump() if isinstance(tests, TestResult) else {}

    structured = fixer_llm.with_structured_output(CodeDraft)

    fix_prompt = (
        "Here is the current implementation:\n\n"
        f"{code}\n\n"
        "Test results (JSON):\n"
        f"{json.dumps(tests_json, indent=2)}\n\n"
        "Please improve the code to satisfy all tests and keep it clean and idiomatic."
    )

    draft: CodeDraft = structured.invoke(
        [
            SystemMessage(content=FIXER_SYSTEM_PROMPT),
            HumanMessage(content=fix_prompt),
        ]
    )

    new_iter = state.get("iteration", 0) + 1

    new_messages: List[BaseMessage] = list(state["messages"]) + [
        HumanMessage(content=f"[fixer] produced revision {new_iter}")
    ]

    return {
        "messages": new_messages,
        "code": draft.code,
        "draft_rationale": draft.rationale,
        "tests": tests,
        "iteration": new_iter,
        "max_iterations": state.get("max_iterations", 3),
        "done": False,
    }

# Routing logic

In [8]:
def route_after_writer(state: FibState) -> str:
    # Always go to tester after the first draft
    return "tester"


def route_after_tester(state: FibState) -> str:
    tests = state.get("tests")
    iteration = state.get("iteration", 0)
    max_iterations = state.get("max_iterations", 3)

    if isinstance(tests, TestResult) and tests.passed:
        return "done"
    if iteration >= max_iterations:
        return "done"
    return "fixer"


def done_node(state: FibState) -> FibState:
    """
    Final node. Just marks the state as done.
    """
    return {
        **state,
        "done": True,
    }


# Build and run the graph

In [9]:
def build_app(sandbox: Sandbox):
    graph = StateGraph(FibState)

    # Wrap tester so sandbox is closed over
    def tester_with_sb(st: FibState) -> FibState:
        return tester_node(st, sandbox)

    graph.add_node("writer", writer_node)
    graph.add_node("tester", tester_with_sb)
    graph.add_node("fixer", fixer_node)
    graph.add_node("done", done_node)

    graph.add_edge(START, "writer")
    graph.add_conditional_edges("writer", route_after_writer, {"tester": "tester"})
    graph.add_conditional_edges(
        "tester",
        route_after_tester,
        {"fixer": "fixer", "done": "done"},
    )
    graph.add_edge("fixer", "tester")  # loop fixer → tester
    graph.add_edge("done", END)

    return graph.compile()


if __name__ == "__main__":
    # Single sandbox for the whole run
    sb = create_sandbox()
    try:
        app = build_app(sb)

        initial_state: FibState = {
            "messages": [
                HumanMessage(
                    content="Write an efficient Fibonacci implementation fib(n: int) -> int."
                )
            ],
            "iteration": 0,
            "max_iterations": 3,
        }

        final_state = app.invoke(initial_state)

        print("Done:", final_state.get("done"))
        print("Iterations:", final_state.get("iteration"))
        print("\nFinal code:\n")
        print(final_state.get("code"))
        print("\nRationale:\n")
        print(final_state.get("draft_rationale"))

        tests = final_state.get("tests")
        if isinstance(tests, TestResult):
            print("\nTests passed:", tests.passed)
            print("Details:", tests.details)
            if tests.failed_cases:
                print("Failed cases:", tests.failed_cases)

    finally:
        sb.kill()


/usr/local/lib/python3.12/dist-packages/langchain_openai/chat_models/base.py:2054: UserWarning: Received a Pydantic BaseModel V1 schema. This is not supported by method="json_schema". Please use method="function_calling" or specify schema via JSON Schema or Pydantic V2 BaseModel. Overriding to method="function_calling".
  warnings.warn(


Done: True
Iterations: 0

Final code:

def fib(n: int) -> int:
    if n == 0:
        return 0
    a, b = 0, 1
    for _ in range(1, n):
        a, b = b, a + b
    return b


Rationale:

The function fib uses an iterative approach to compute the nth Fibonacci number efficiently. It handles the base case n=0 by returning 0 immediately. For n > 0, it iterates from 1 to n-1, updating two variables a and b to hold consecutive Fibonacci numbers. This approach has O(n) time complexity and O(1) space complexity, avoiding the exponential time and stack overhead of naive recursion.

Tests passed: True
Details: Tests completed
